# Raman Data Cleaning - Excel File Combiner

**Purpose:** Combine 16 individual Excel files into one clean dataset

**Input:** 16 Excel files with Raman shift (column D) and dark-subtracted intensity (column H)

**Output:** `output.xlsx` with columns: X1 | Y1 | Y2 | Y3 | ... | Y16

---

## Workflow:
1. Extract **X1** (Raman shift) from column D, rows 100-2147 of **FIRST file only**
2. Extract **Y1-Y16** (intensity) from column H, rows 100-2147 of **ALL 16 files**
3. Combine into single Excel file with proper alignment

## Step 1: Import Required Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## Step 2: Set Folder Path

**⚠️ CHANGE THIS PATH to your data folder**

In [ ]:
data_folder = Path(
    r"c:\Users\sukes\Downloads\sukesh NAM Raman data\data of Dry powder\5sec-power5-80 and 2set"
)

print(f"📁 Data folder: {data_folder}")
print(f"   Folder exists: {data_folder.exists()}")

## Step 3: Collect All Excel Files

Find exactly 16 Excel files in the folder

In [ ]:
# Collect all Excel files, excluding temporary files (~$)
excel_files = sorted([f for f in data_folder.glob("*.xlsx") if not f.name.startswith('~$')])

print(f"📊 Found {len(excel_files)} Excel files:")
for i, file in enumerate(excel_files, 1):
    print(f"   {i:2d}. {file.name}")

if len(excel_files) != 16:
    print(f"\n⚠️ WARNING: Expected 16 files, found {len(excel_files)}")
else:
    print(f"\n✅ Correct number of files found!")

## Step 4: Extract X1 (Raman Shift) from First File Only

- **Column:** D (index 3)
- **Rows:** 100 to 2147 (Python indices: 99 to 2147)
- **Source:** First file only

In [ ]:
# Read first file
df_first = pd.read_excel(excel_files[0], header=None)

# Column D = index 3, rows 100–2147 → Python indices 99:2147
x1 = df_first.iloc[99:2147, 3].values

print(f"✅ Extracted X1 (Raman shift)")
print(f"   Source file: {excel_files[0].name}")
print(f"   Column: D")
print(f"   Rows: 100-2147")
print(f"   Data points: {len(x1)}")
print(f"   Range: {x1.min():.2f} to {x1.max():.2f} cm⁻¹")

## Step 5: Extract Y1-Y16 (Dark-Subtracted Intensity) from All Files

- **Column:** H (index 7)
- **Rows:** 100 to 2147 (Python indices: 99 to 2147)
- **Source:** All 16 files

In [ ]:
Y_list = []

for i, file in enumerate(excel_files[:16], 1):
    df = pd.read_excel(file, header=None)
    
    # Column H = index 7, rows 100–2147
    y = df.iloc[99:2147, 7].values
    Y_list.append(y)
    
    print(f"✅ Y{i:2d} extracted from {file.name}")

print(f"\n📊 Summary:")
print(f"   Total Y columns: {len(Y_list)}")
print(f"   Data points per column: {len(Y_list[0])}")

## Step 6: Create Combined DataFrame

Combine X1 and all Y columns into a single DataFrame

In [ ]:
# Convert Y_list to 2D array (rows × spectra)
Y = np.array(Y_list).T

# Create column names
columns = ["X1"] + [f"Y{i+1}" for i in range(Y.shape[1])]

# Combine X1 and Y columns
data = np.column_stack([x1, Y])

# Create DataFrame
df_output = pd.DataFrame(data, columns=columns)

print(f"✅ Combined DataFrame created")
print(f"   Shape: {df_output.shape} (rows × columns)")
print(f"   Columns: {list(df_output.columns)}")
print(f"\n📊 Preview:")
print(df_output.head())

## Step 7: Save to Excel File

Save the combined data to `output.xlsx`

In [ ]:
output_file = data_folder / "output.xlsx"

df_output.to_excel(output_file, index=False)

print("✅ output.xlsx created successfully!")
print(f"   Location: {output_file}")
print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")
print(f"\n📋 Final Structure:")
print(f"   • X1: Raman shift (cm⁻¹) from column D, rows 100-2147")
print(f"   • Y1-Y16: Dark-subtracted intensity from column H, rows 100-2147")
print(f"   • Total data points: {len(df_output)}")
print(f"   • Total spectra: {len(df_output.columns) - 1}")

---

## ✅ Data Cleaning Complete!

### Output File Structure:

| Column | Source | Description |
|--------|--------|-------------|
| **X1** | First file, Column D, Rows 100-2147 | Raman shift (cm⁻¹) |
| **Y1** | File 1, Column H, Rows 100-2147 | Dark-subtracted intensity |
| **Y2** | File 2, Column H, Rows 100-2147 | Dark-subtracted intensity |
| **...** | ... | ... |
| **Y16** | File 16, Column H, Rows 100-2147 | Dark-subtracted intensity |

### Next Steps:
1. ✅ Use `output.xlsx` in the main analysis notebook
2. ✅ Run baseline correction, smoothing, and normalization
3. ✅ Perform peak detection and stability analysis

### To Process Other Integration Times:
1. Change `data_folder` path to another folder (10sec, 15sec, 20sec, 25sec)
2. Run all cells again
3. Rename output file to match integration time (e.g., `10sec-data.xlsx`)

---

## 🔄 Process All Integration Times Automatically

Process all folders (10sec, 15sec, 20sec, 25sec) in one go

In [6]:
# Define all folders to process
base_folder = Path(r"c:\Users\sukes\Downloads\sukesh NAM Raman data\data of Dry powder")

folders_to_process = [
    base_folder / "5sec-power5-80 and 2set",
    base_folder / "10sec-power5-80-2set",
    base_folder / "15sec-power5-80andset2",
    base_folder / "20sec-power5-80and set2",
    base_folder / "25sec-power5-80 and set2"
]

print("📁 Folders to process:")
for i, folder in enumerate(folders_to_process, 1):
    status = "✅ Exists" if folder.exists() else "❌ Not found"
    # Count files in folder
    if folder.exists():
        csv_count = len(list(folder.glob("*.csv")))
        excel_count = len([f for f in folder.glob("*.xlsx") if not f.name.startswith('~$')])
        print(f"   {i}. {folder.name} - {status} ({csv_count} CSV, {excel_count} Excel)")
    else:
        print(f"   {i}. {folder.name} - {status}")

📁 Folders to process:
   1. 5sec-power5-80 and 2set - ✅ Exists (16 CSV, 1 Excel)
   2. 10sec-power5-80-2set - ✅ Exists (17 CSV, 18 Excel)
   3. 15sec-power5-80andset2 - ✅ Exists (16 CSV, 17 Excel)
   4. 20sec-power5-80and set2 - ✅ Exists (16 CSV, 17 Excel)
   5. 25sec-power5-80 and set2 - ✅ Exists (16 CSV, 17 Excel)


In [7]:
# Process each folder (works with both CSV and Excel files)
results = []

for folder in folders_to_process:
    if not folder.exists():
        print(f"\n⚠️ Skipping {folder.name} - folder not found")
        continue
    
    print(f"\n{'='*70}")
    print(f"📂 Processing: {folder.name}")
    print('='*70)
    
    # First check for Excel files
    excel_files = sorted([f for f in folder.glob("*.xlsx") if not f.name.startswith('~$') and f.name != 'output.xlsx'])
    
    # If no Excel files, look for CSV files
    if len(excel_files) == 0:
        print(f"   No Excel files found, looking for CSV files...")
        csv_files = sorted(folder.glob("SP_*.csv"))
        
        if len(csv_files) == 0:
            print(f"   ⚠️ No CSV or Excel files found - skipping")
            continue
        
        print(f"   Found {len(csv_files)} CSV files - converting to Excel...")
        
        # Convert CSV to Excel (read from proper rows)
        excel_files = []
        for csv_file in csv_files:
            try:
                # Read CSV with proper parameters
                df_csv = pd.read_csv(csv_file, skiprows=98, nrows=2048, header=None)
                
                # Create Excel file
                excel_file = csv_file.with_suffix('.xlsx')
                df_csv.to_excel(excel_file, index=False, header=False)
                excel_files.append(excel_file)
                
            except Exception as e:
                print(f"   ⚠️ Error converting {csv_file.name}: {e}")
        
        print(f"   ✅ Converted {len(excel_files)} CSV files to Excel")
    
    print(f"   Processing {len(excel_files)} Excel files")
    
    if len(excel_files) == 0:
        print(f"   ⚠️ No files to process - skipping")
        continue
    
    try:
        # Extract X1 from first file
        df_first = pd.read_excel(excel_files[0], header=None)
        x1 = df_first.iloc[:, 3].values  # Column D (index 3)
        
        # Extract Y1-Y16 from all files
        Y_list = []
        for file in excel_files[:16]:
            df = pd.read_excel(file, header=None)
            y = df.iloc[:, 7].values  # Column H (index 7)
            Y_list.append(y)
        
        # Create combined DataFrame
        Y = np.array(Y_list).T
        columns = ["X1"] + [f"Y{i+1}" for i in range(Y.shape[1])]
        data = np.column_stack([x1, Y])
        df_output = pd.DataFrame(data, columns=columns)
        
        # Save to output file IN THE SAME FOLDER
        output_file = folder / "output.xlsx"
        df_output.to_excel(output_file, index=False)
        
        results.append({
            'folder': folder.name,
            'files_processed': len(Y_list),
            'data_points': len(x1),
            'output_file': output_file,
            'status': 'SUCCESS'
        })
        
        print(f"   ✅ SUCCESS!")
        print(f"   • Processed {len(Y_list)} files")
        print(f"   • Data points: {len(x1)}")
        print(f"   • Output saved: {output_file.name}")
        
    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        results.append({
            'folder': folder.name,
            'status': 'FAILED',
            'error': str(e)
        })

print(f"\n{'='*70}")
print("📊 PROCESSING SUMMARY")
print('='*70)
for result in results:
    if result['status'] == 'SUCCESS':
        print(f"\n✅ {result['folder']}")
        print(f"   Files: {result['files_processed']}")
        print(f"   Points: {result['data_points']}")
        print(f"   Output: {result['output_file']}")
    else:
        print(f"\n❌ {result['folder']}")
        print(f"   Error: {result.get('error', 'Unknown error')}")

print(f"\n{'='*70}")
print(f"🎯 Total folders processed: {len([r for r in results if r['status'] == 'SUCCESS'])}/{len(folders_to_process)}")
print('='*70)


📂 Processing: 5sec-power5-80 and 2set
   Processing 1 Excel files
   ✅ SUCCESS!
   • Processed 1 files
   • Data points: 2049
   • Output saved: output.xlsx

📂 Processing: 10sec-power5-80-2set
   Processing 17 Excel files
   ❌ ERROR: [Errno 13] Permission denied: 'c:\\Users\\sukes\\Downloads\\sukesh NAM Raman data\\data of Dry powder\\10sec-power5-80-2set\\output.xlsx'

📂 Processing: 15sec-power5-80andset2
   Processing 16 Excel files
   ✅ SUCCESS!
   • Processed 16 files
   • Data points: 2048
   • Output saved: output.xlsx

📂 Processing: 20sec-power5-80and set2
   Processing 16 Excel files
   ✅ SUCCESS!
   • Processed 16 files
   • Data points: 2048
   • Output saved: output.xlsx

📂 Processing: 25sec-power5-80 and set2
   Processing 16 Excel files
   ✅ SUCCESS!
   • Processed 16 files
   • Data points: 2048
   • Output saved: output.xlsx

📊 PROCESSING SUMMARY

✅ 5sec-power5-80 and 2set
   Files: 1
   Points: 2049
   Output: c:\Users\sukes\Downloads\sukesh NAM Raman data\data of Dry 